# 🏆 PS S6E3 — Telco Customer Churn — Top 10 Pipeline

**Sections:**
1. Environment & Config
2. Data Loading
3. Preprocessing
4. Feature Engineering
5. Model Training (OOF Pipeline)
6. Ensembling (Weighted Average + Meta-Model)
7. Hill Climbing
8. Submission

## 1. Environment & Config

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
import gc
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

# ============ CONFIG ============
N_FOLDS = 10
SEED = 42
TARGET = "Churn"
DROP_COLS = ["id", "customerID"]

np.random.seed(SEED)
print("✅ STEP 1 COMPLETE — Environment Ready")

✅ STEP 1 COMPLETE — Environment Ready


## 2. Data Loading (UNTOUCHED)

In [2]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
original = pd.read_csv(ORIG_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Original shape:", original.shape)

Train shape: (594194, 21)
Test shape: (254655, 20)
Original shape: (7043, 21)


## 3. Preprocessing

In [4]:
# --- Clean original dataset to match train columns ---
original["Churn"] = (original["Churn"] == "Yes").astype(int)
original["TotalCharges"] = pd.to_numeric(original["TotalCharges"], errors="coerce")

if "id" not in original.columns:
    original["id"] = range(len(train), len(train) + len(original))

original = original[train.columns]
print("Original cleaned successfully")

Original cleaned successfully


In [5]:
# --- De-duplicate original vs train ---
cols = [c for c in train.columns if c not in ["id", "customerID"]]
train_hash = train[cols].apply(lambda row: hash(tuple(row)), axis=1)
orig_hash  = original[cols].apply(lambda row: hash(tuple(row)), axis=1)

is_duplicate = orig_hash.isin(train_hash)
print("Number of duplicates found:", is_duplicate.sum())
original_clean = original[~is_duplicate].reset_index(drop=True)

Number of duplicates found: 0


In [6]:
# --- Combine train + cleaned original ---
train_full = pd.concat([train, original_clean], ignore_index=True)
print("Combined train shape:", train_full.shape)
print("Test shape:", test.shape)

Combined train shape: (601237, 21)
Test shape: (254655, 20)


In [7]:
def preprocess(df):
    df = df.copy()
    df["gender"] = (df["gender"] == "Male").astype(int)

    binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] == "Yes").astype(int)

    df["Contract"] = df["Contract"].map({
        "Month-to-month": 0, "One year": 1, "Two year": 2
    })
    df["InternetService"] = df["InternetService"].map({
        "No": 0, "DSL": 1, "Fiber optic": 2
    })
    df["PaymentMethod"] = df["PaymentMethod"].map({
        "Electronic check": 0, "Mailed check": 1,
        "Bank transfer (automatic)": 2, "Credit card (automatic)": 3
    })

    service_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies", "MultipleLines"
    ]
    for col in service_cols:
        if col in df.columns:
            df[col] = (df[col] == "Yes").astype(int)

    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["tenure"] * df["MonthlyCharges"])
    return df

train_full = preprocess(train_full)
test = preprocess(test)

# Fix target
train_full[TARGET] = train_full[TARGET].replace({"Yes": 1, "No": 0}).astype(int)
print("Preprocessing done")
print("Target distribution:\n", train_full[TARGET].value_counts())

Preprocessing done
Target distribution:
 Churn
0    465551
1    135686
Name: count, dtype: int64


## 4. Feature Engineering

In [8]:
def add_features(df):
    df = df.copy()
    eps = 1e-6

    # --- Basic ratio features ---
    df["charges_per_month"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["charge_diff"] = df["MonthlyCharges"] - df["charges_per_month"]
    df["avg_charge_increase"] = df["charge_diff"] / (df["tenure"] + 1)
    df["monthly_to_total_ratio"] = df["MonthlyCharges"] / (df["TotalCharges"] + eps)
    df["expected_total"] = df["MonthlyCharges"] * df["tenure"]
    df["total_charge_deviation"] = df["TotalCharges"] - df["expected_total"]

    # --- Log transforms ---
    df["log_monthly_charges"] = np.log1p(df["MonthlyCharges"])
    df["log_total_charges"] = np.log1p(df["TotalCharges"])
    df["log_tenure"] = np.log1p(df["tenure"])

    # --- Service features ---
    service_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies", "MultipleLines"
    ]
    df["service_count"] = df[service_cols].sum(axis=1)
    df["protection_count"] = df[["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]].sum(axis=1)
    df["streaming_count"] = df[["StreamingTV", "StreamingMovies"]].sum(axis=1)
    df["has_protection"] = (df["protection_count"] > 0).astype(int)
    df["has_streaming"] = (df["streaming_count"] > 0).astype(int)
    df["charge_per_service"] = df["MonthlyCharges"] / (df["service_count"] + 1)

    # --- Tenure features ---
    df["is_new_customer"] = (df["tenure"] <= 3).astype(int)
    df["is_loyal_customer"] = (df["tenure"] >= 24).astype(int)
    df["is_very_loyal"] = (df["tenure"] >= 48).astype(int)
    df["tenure_bin"] = pd.cut(
        df["tenure"],
        bins=[-1, 3, 6, 12, 24, 48, 72, 999],
        labels=[0, 1, 2, 3, 4, 5, 6]
    ).astype(float)

    # --- Rank features ---
    df["monthly_charge_rank"] = df["MonthlyCharges"].rank(pct=True)
    df["total_charge_rank"] = df["TotalCharges"].rank(pct=True)
    df["tenure_rank"] = df["tenure"].rank(pct=True)

    # --- Payment / contract features ---
    df["auto_pay"] = (df["PaymentMethod"] >= 2).astype(int)
    df["electronic_check"] = (df["PaymentMethod"] == 0).astype(int)

    # --- Internet features ---
    df["no_internet"] = (df["InternetService"] == 0).astype(int)
    df["has_fiber"] = (df["InternetService"] == 2).astype(int)

    # --- Interaction features ---
    df["tenure_x_contract"] = df["tenure"] * df["Contract"]
    df["monthly_x_contract"] = df["MonthlyCharges"] * df["Contract"]
    df["tenure_x_monthly"] = df["tenure"] * df["MonthlyCharges"]
    df["tenure_x_services"] = df["tenure"] * df["service_count"]
    df["contract_x_services"] = df["Contract"] * df["service_count"]
    df["tenure_x_autopay"] = df["tenure"] * df["auto_pay"]
    df["monthly_x_partner"] = df["MonthlyCharges"] * df["Partner"]
    df["tenure_x_dependents"] = df["tenure"] * df["Dependents"]
    df["fiber_x_tenure"] = df["has_fiber"] * df["tenure"]
    df["fiber_x_services"] = df["has_fiber"] * df["service_count"]

    # --- Risk indicator features ---
    df["fiber_no_security"]  = ((df["InternetService"]==2) & (df["OnlineSecurity"]==0)).astype(int)
    df["new_month_to_month"]  = ((df["Contract"]==0) & (df["tenure"]<=6)).astype(int)
    df["high_value_no_contract"] = ((df["MonthlyCharges"] > 70) & (df["Contract"] == 0)).astype(int)
    df["fiber_no_protection"] = ((df["InternetService"]==2) & (df["protection_count"]==0)).astype(int)
    df["echeck_month_to_month"] = ((df["PaymentMethod"]==0) & (df["Contract"]==0)).astype(int)
    df["new_fiber_echeck"] = ((df["tenure"]<=6) & (df["InternetService"]==2) & (df["PaymentMethod"]==0)).astype(int)

    # --- Aggregated value features ---
    df["total_service_value"] = df["TotalCharges"] * df["service_count"]
    df["lifetime_value"] = df["MonthlyCharges"] * df["tenure"] * (df["Contract"] + 1)

    # --- SeniorCitizen interactions ---
    df["senior_alone"] = ((df["SeniorCitizen"]==1) & (df["Partner"]==0) & (df["Dependents"]==0)).astype(int)
    df["senior_no_security"] = ((df["SeniorCitizen"]==1) & (df["OnlineSecurity"]==0) & (df["TechSupport"]==0)).astype(int)

    return df

train_full = add_features(train_full)
test = add_features(test)

FEATURES = [c for c in train_full.columns if c not in DROP_COLS + [TARGET]]
print(f"Total features: {len(FEATURES)}")
print("Sample features:", FEATURES[:10])

Total features: 65
Sample features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup']


## 5. Model Training — OOF Pipeline

All models produce:
- `oof` predictions on train
- `test_preds` averaged across folds

We train **9 diverse models** (3 base × 3 seeds/configs).

In [9]:
# ============ GENERIC OOF TRAINER ============
def train_model_oof(model_fn, model_name, train_df, test_df, features, target, folds):
    """
    Generic OOF training function.
    model_fn: callable(fold) -> model with .fit() and .predict_proba()
    Returns: oof_preds, test_preds, cv_score
    """
    oof = np.zeros(len(train_df))
    test_preds = np.zeros(len(test_df))
    n_folds = len(folds)

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        X_tr = train_df.iloc[tr_idx][features]
        y_tr = train_df.iloc[tr_idx][target]
        X_val = train_df.iloc[val_idx][features]
        y_val = train_df.iloc[val_idx][target]

        model = model_fn(fold_i)

        if isinstance(model, lgb.LGBMClassifier):
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False)]
            )
        elif isinstance(model, xgb.XGBClassifier):
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        elif isinstance(model, CatBoostClassifier):
            model.fit(
                X_tr, y_tr,
                eval_set=(X_val, y_val),
                verbose=False
            )
        else:
            model.fit(X_tr, y_tr)

        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test_df[features])[:, 1] / n_folds

        fold_auc = roc_auc_score(y_val, oof[val_idx])
        print(f"  {model_name} Fold {fold_i}: AUC = {fold_auc:.6f}")

        del model; gc.collect()

    cv_score = roc_auc_score(train_df[target], oof)
    print(f">>> {model_name} CV AUC = {cv_score:.6f}\n")
    return oof, test_preds, cv_score

print("OOF trainer ready")

OOF trainer ready


In [10]:
# ============ FOLDS ============
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train_full, train_full[TARGET]))

for i, (tr, val) in enumerate(FOLDS):
    tr_mean = train_full.iloc[tr][TARGET].mean()
    val_mean = train_full.iloc[val][TARGET].mean()
    print(f"Fold {i}: train churn={tr_mean:.4f}, val churn={val_mean:.4f}, val_size={len(val)}")

Fold 0: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 1: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 2: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 3: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 4: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 5: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 6: train churn=0.2257, val churn=0.2257, val_size=60124
Fold 7: train churn=0.2257, val churn=0.2257, val_size=60123
Fold 8: train churn=0.2257, val churn=0.2257, val_size=60123
Fold 9: train churn=0.2257, val churn=0.2257, val_size=60123


In [11]:
# ============ STORAGE ============
oof_dict = {}   # model_name -> oof_preds
test_dict = {}  # model_name -> test_preds
cv_scores = {}  # model_name -> cv_score

print("Storage initialized")

Storage initialized


### 5A. LightGBM Models (3 configs)

In [12]:
# --- LGB Config 1 ---
def lgb_model_1(fold_i):
    return lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, num_leaves=31,
        max_depth=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, min_child_samples=20,
        random_state=SEED, verbose=-1, n_jobs=-1
    )

oof, test_p, score = train_model_oof(lgb_model_1, "LGB_v1", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["LGB_v1"] = oof; test_dict["LGB_v1"] = test_p; cv_scores["LGB_v1"] = score

  LGB_v1 Fold 0: AUC = 0.915660
  LGB_v1 Fold 1: AUC = 0.916125
  LGB_v1 Fold 2: AUC = 0.918279
  LGB_v1 Fold 3: AUC = 0.914271
  LGB_v1 Fold 4: AUC = 0.914710
  LGB_v1 Fold 5: AUC = 0.913532
  LGB_v1 Fold 6: AUC = 0.915224
  LGB_v1 Fold 7: AUC = 0.915257
  LGB_v1 Fold 8: AUC = 0.916065
  LGB_v1 Fold 9: AUC = 0.915936
>>> LGB_v1 CV AUC = 0.915499



In [13]:
# --- LGB Config 2 (deeper, different seed) ---
def lgb_model_2(fold_i):
    return lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.015, num_leaves=63,
        max_depth=7, subsample=0.75, colsample_bytree=0.7,
        reg_alpha=0.5, reg_lambda=2.0, min_child_samples=30,
        random_state=SEED + 1, verbose=-1, n_jobs=-1
    )

oof, test_p, score = train_model_oof(lgb_model_2, "LGB_v2", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["LGB_v2"] = oof; test_dict["LGB_v2"] = test_p; cv_scores["LGB_v2"] = score

  LGB_v2 Fold 0: AUC = 0.915684
  LGB_v2 Fold 1: AUC = 0.916187
  LGB_v2 Fold 2: AUC = 0.918624
  LGB_v2 Fold 3: AUC = 0.914637
  LGB_v2 Fold 4: AUC = 0.914899
  LGB_v2 Fold 5: AUC = 0.913526
  LGB_v2 Fold 6: AUC = 0.915474
  LGB_v2 Fold 7: AUC = 0.915338
  LGB_v2 Fold 8: AUC = 0.916182
  LGB_v2 Fold 9: AUC = 0.916115
>>> LGB_v2 CV AUC = 0.915657



In [14]:
# --- LGB Config 3 (shallow, regularized) ---
def lgb_model_3(fold_i):
    return lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.008, num_leaves=16,
        max_depth=4, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=1.0, reg_lambda=5.0, min_child_samples=50,
        random_state=SEED + 2, verbose=-1, n_jobs=-1
    )

oof, test_p, score = train_model_oof(lgb_model_3, "LGB_v3", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["LGB_v3"] = oof; test_dict["LGB_v3"] = test_p; cv_scores["LGB_v3"] = score

  LGB_v3 Fold 0: AUC = 0.915112
  LGB_v3 Fold 1: AUC = 0.915693
  LGB_v3 Fold 2: AUC = 0.918019
  LGB_v3 Fold 3: AUC = 0.913678
  LGB_v3 Fold 4: AUC = 0.914157
  LGB_v3 Fold 5: AUC = 0.912976
  LGB_v3 Fold 6: AUC = 0.914773
  LGB_v3 Fold 7: AUC = 0.914730
  LGB_v3 Fold 8: AUC = 0.915631
  LGB_v3 Fold 9: AUC = 0.915275
>>> LGB_v3 CV AUC = 0.914997



### 5B. XGBoost Models (3 configs)

In [15]:
# --- XGB Config 1 ---
def xgb_model_1(fold_i):
    return xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        eval_metric="auc", early_stopping_rounds=100,
        random_state=SEED, verbosity=0, tree_method="hist",
        device="cuda"
    )

oof, test_p, score = train_model_oof(xgb_model_1, "XGB_v1", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["XGB_v1"] = oof; test_dict["XGB_v1"] = test_p; cv_scores["XGB_v1"] = score

  XGB_v1 Fold 0: AUC = 0.915801
  XGB_v1 Fold 1: AUC = 0.916175
  XGB_v1 Fold 2: AUC = 0.918620
  XGB_v1 Fold 3: AUC = 0.914527
  XGB_v1 Fold 4: AUC = 0.914853
  XGB_v1 Fold 5: AUC = 0.913647
  XGB_v1 Fold 6: AUC = 0.915610
  XGB_v1 Fold 7: AUC = 0.915369
  XGB_v1 Fold 8: AUC = 0.916261
  XGB_v1 Fold 9: AUC = 0.916090
>>> XGB_v1 CV AUC = 0.915689



In [16]:
# --- XGB Config 2 (deeper, different seed) ---
def xgb_model_2(fold_i):
    return xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=7,
        min_child_weight=30, subsample=0.75, colsample_bytree=0.7,
        gamma=0.2, reg_alpha=0.5, reg_lambda=3.0,
        eval_metric="auc", early_stopping_rounds=100,
        random_state=SEED + 1, verbosity=0, tree_method="hist",
        device="cuda"
    )

oof, test_p, score = train_model_oof(xgb_model_2, "XGB_v2", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["XGB_v2"] = oof; test_dict["XGB_v2"] = test_p; cv_scores["XGB_v2"] = score

  XGB_v2 Fold 0: AUC = 0.915903
  XGB_v2 Fold 1: AUC = 0.916224
  XGB_v2 Fold 2: AUC = 0.918713
  XGB_v2 Fold 3: AUC = 0.914785
  XGB_v2 Fold 4: AUC = 0.915033
  XGB_v2 Fold 5: AUC = 0.913798
  XGB_v2 Fold 6: AUC = 0.915755
  XGB_v2 Fold 7: AUC = 0.915422
  XGB_v2 Fold 8: AUC = 0.916428
  XGB_v2 Fold 9: AUC = 0.916259
>>> XGB_v2 CV AUC = 0.915824



In [17]:
# --- XGB Config 3 (shallow, conservative) ---
def xgb_model_3(fold_i):
    return xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.008, max_depth=4,
        min_child_weight=50, subsample=0.85, colsample_bytree=0.85,
        gamma=0.3, reg_alpha=1.0, reg_lambda=5.0,
        eval_metric="auc", early_stopping_rounds=100,
        random_state=SEED + 2, verbosity=0, tree_method="hist",
        device="cuda"
    )

oof, test_p, score = train_model_oof(xgb_model_3, "XGB_v3", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["XGB_v3"] = oof; test_dict["XGB_v3"] = test_p; cv_scores["XGB_v3"] = score

  XGB_v3 Fold 0: AUC = 0.915220
  XGB_v3 Fold 1: AUC = 0.915825
  XGB_v3 Fold 2: AUC = 0.918009
  XGB_v3 Fold 3: AUC = 0.913782
  XGB_v3 Fold 4: AUC = 0.914223
  XGB_v3 Fold 5: AUC = 0.913018
  XGB_v3 Fold 6: AUC = 0.914949
  XGB_v3 Fold 7: AUC = 0.914799
  XGB_v3 Fold 8: AUC = 0.915699
  XGB_v3 Fold 9: AUC = 0.915454
>>> XGB_v3 CV AUC = 0.915089



### 5C. CatBoost Models (3 configs)

In [18]:
# --- CAT Config 1 ---
def cat_model_1(fold_i):
    return CatBoostClassifier(
        iterations=3000, learning_rate=0.03, depth=6,
        l2_leaf_reg=3, bagging_temperature=0.5,
        eval_metric="AUC", od_type="Iter", od_wait=100,
        random_seed=SEED, verbose=False, task_type="GPU"
    )

oof, test_p, score = train_model_oof(cat_model_1, "CAT_v1", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["CAT_v1"] = oof; test_dict["CAT_v1"] = test_p; cv_scores["CAT_v1"] = score

Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v1 Fold 0: AUC = 0.915500


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v1 Fold 1: AUC = 0.915979


Default metric period is 5 because AUC is/are not implemented for GPU


...

  CAT_v1 Fold 7: AUC = 0.915278


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v1 Fold 8: AUC = 0.915930


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v1 Fold 9: AUC = 0.915699
>>> CAT_v1 CV AUC = 0.915410



In [19]:
# --- CAT Config 2 (deeper, different seed) ---
def cat_model_2(fold_i):
    return CatBoostClassifier(
        iterations=3000, learning_rate=0.05, depth=8,
        l2_leaf_reg=5, bagging_temperature=1.0,
        eval_metric="AUC", od_type="Iter", od_wait=100,
        random_seed=SEED + 1, verbose=False, task_type="GPU"
    )

oof, test_p, score = train_model_oof(cat_model_2, "CAT_v2", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["CAT_v2"] = oof; test_dict["CAT_v2"] = test_p; cv_scores["CAT_v2"] = score

Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v2 Fold 0: AUC = 0.915115


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v2 Fold 1: AUC = 0.915615


Default metric period is 5 because AUC is/are not implemented for GPU


...

  CAT_v2 Fold 7: AUC = 0.914750


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v2 Fold 8: AUC = 0.915589


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v2 Fold 9: AUC = 0.915366
>>> CAT_v2 CV AUC = 0.914985



In [20]:
# --- CAT Config 3 (shallow, regularized) ---
def cat_model_3(fold_i):
    return CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=4,
        l2_leaf_reg=10, bagging_temperature=0.3,
        eval_metric="AUC", od_type="Iter", od_wait=100,
        random_seed=SEED + 2, verbose=False, task_type="GPU"
    )

oof, test_p, score = train_model_oof(cat_model_3, "CAT_v3", train_full, test, FEATURES, TARGET, FOLDS)
oof_dict["CAT_v3"] = oof; test_dict["CAT_v3"] = test_p; cv_scores["CAT_v3"] = score

Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v3 Fold 0: AUC = 0.915210


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v3 Fold 1: AUC = 0.915664


Default metric period is 5 because AUC is/are not implemented for GPU


...

  CAT_v3 Fold 7: AUC = 0.914884


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v3 Fold 8: AUC = 0.915647


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT_v3 Fold 9: AUC = 0.915389
>>> CAT_v3 CV AUC = 0.915048



In [21]:
# ============ SUMMARY ============
print("\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)
for name, sc in sorted(cv_scores.items(), key=lambda x: -x[1]):
    print(f"  {name:15s} CV AUC = {sc:.6f}")
print("="*60)


MODEL SUMMARY
  XGB_v2          CV AUC = 0.915824
  XGB_v1          CV AUC = 0.915689
  LGB_v2          CV AUC = 0.915657
  LGB_v1          CV AUC = 0.915499
  CAT_v1          CV AUC = 0.915410
  XGB_v3          CV AUC = 0.915089
  CAT_v3          CV AUC = 0.915048
  LGB_v3          CV AUC = 0.914997
  CAT_v2          CV AUC = 0.914985


## 6. Ensembling

### 6A. Rank-Averaged Weighted Ensemble

In [22]:
def rank_avg(preds):
    return rankdata(preds) / len(preds)

# --- Method 1: Equal-weight rank average ---
model_names = list(oof_dict.keys())

ensemble_oof = np.zeros(len(train_full))
ensemble_test = np.zeros(len(test))
for name in model_names:
    ensemble_oof += rank_avg(oof_dict[name])
    ensemble_test += rank_avg(test_dict[name])
ensemble_oof /= len(model_names)
ensemble_test /= len(model_names)

print(f"Equal-weight Rank Ensemble CV AUC: {roc_auc_score(train_full[TARGET], ensemble_oof):.6f}")

Equal-weight Rank Ensemble CV AUC: 0.915674


In [23]:
# --- Method 2: Optimized weight search (Nelder-Mead) ---
from scipy.optimize import minimize

def neg_auc_weighted(weights):
    w = np.abs(weights)
    w = w / w.sum()
    blend = np.zeros(len(train_full))
    for i, name in enumerate(model_names):
        blend += w[i] * rank_avg(oof_dict[name])
    return -roc_auc_score(train_full[TARGET], blend)

initial_weights = np.ones(len(model_names)) / len(model_names)
result = minimize(neg_auc_weighted, initial_weights, method="Nelder-Mead",
                  options={"maxiter": 10000, "xatol": 1e-8, "fatol": 1e-8})

opt_weights = np.abs(result.x)
opt_weights = opt_weights / opt_weights.sum()

print("\nOptimized weights:")
for name, w in zip(model_names, opt_weights):
    print(f"  {name:15s}: {w:.4f}")

# Apply optimized weights
opt_oof = np.zeros(len(train_full))
opt_test = np.zeros(len(test))
for i, name in enumerate(model_names):
    opt_oof += opt_weights[i] * rank_avg(oof_dict[name])
    opt_test += opt_weights[i] * rank_avg(test_dict[name])

print(f"\nOptimized Weighted Ensemble CV AUC: {roc_auc_score(train_full[TARGET], opt_oof):.6f}")


Optimized weights:
  LGB_v1         : 0.0879
  LGB_v2         : 0.2467
  LGB_v3         : 0.0000
  XGB_v1         : 0.2026
  XGB_v2         : 0.3620
  XGB_v3         : 0.0011
  CAT_v1         : 0.0877
  CAT_v2         : 0.0003
  CAT_v3         : 0.0117

Optimized Weighted Ensemble CV AUC: 0.915889


### 6B. Meta-Model Stacking (Ridge + Logistic Regression)

Trained using OOF predictions in a proper CV loop to avoid leakage.

In [24]:
# Build stacking features from OOF predictions
stack_train = pd.DataFrame({name: oof_dict[name] for name in model_names})
stack_test_df = pd.DataFrame({name: test_dict[name] for name in model_names})

print("Stacking features shape:", stack_train.shape)
print(stack_train.head())

Stacking features shape: (601237, 9)
     LGB_v1    LGB_v2    LGB_v3    XGB_v1    XGB_v2    XGB_v3    CAT_v1  \
0  0.010119  0.009128  0.010033  0.009865  0.009530  0.009378  0.010171   
1  0.001411  0.001309  0.001170  0.001084  0.001097  0.001183  0.001421   
2  0.305342  0.293956  0.301906  0.301035  0.277929  0.298949  0.293855   
3  0.752260  0.701082  0.791610  0.745053  0.722628  0.778360  0.711003   
4  0.791741  0.798214  0.795724  0.791310  0.783525  0.803091  0.794139   

     CAT_v2    CAT_v3  
0  0.009731  0.012815  
1  0.001527  0.001706  
2  0.301201  0.294526  
3  0.733176  0.747868  
4  0.795615  0.796067  


In [25]:
# --- Meta-Model 1: Logistic Regression (with proper CV) ---
def train_meta_model_cv(meta_model_fn, stack_train, stack_test, y, folds, name="Meta"):
    meta_oof = np.zeros(len(stack_train))
    meta_test = np.zeros(len(stack_test))
    n_folds = len(folds)

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        X_tr = stack_train.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        X_val = stack_train.iloc[val_idx]

        model = meta_model_fn()
        model.fit(X_tr, y_tr)

        meta_oof[val_idx] = model.predict_proba(X_val)[:, 1]
        meta_test += model.predict_proba(stack_test)[:, 1] / n_folds

    score = roc_auc_score(y, meta_oof)
    print(f"{name} CV AUC: {score:.6f}")
    return meta_oof, meta_test, score

# LR meta-model
def lr_meta():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    )

lr_meta_oof, lr_meta_test, lr_meta_score = train_meta_model_cv(
    lr_meta, stack_train, stack_test_df, train_full[TARGET], FOLDS, "LR_Meta"
)

LR_Meta CV AUC: 0.915902


In [26]:
# --- Meta-Model 2: Ridge Regression ---
from sklearn.linear_model import RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV

def ridge_meta():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.5, penalty='l2', max_iter=1000, random_state=SEED)
    )

ridge_meta_oof, ridge_meta_test, ridge_meta_score = train_meta_model_cv(
    ridge_meta, stack_train, stack_test_df, train_full[TARGET], FOLDS, "Ridge_Meta"
)

Ridge_Meta CV AUC: 0.915902


In [27]:
# --- Meta-Model 3: LGB stacker ---
def lgb_meta():
    return lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=8,
        max_depth=3, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbose=-1, n_jobs=-1
    )

lgb_meta_oof, lgb_meta_test, lgb_meta_score = train_meta_model_cv(
    lgb_meta, stack_train, stack_test_df, train_full[TARGET], FOLDS, "LGB_Meta"
)

LGB_Meta CV AUC: 0.915789


## 7. Hill Climbing — Greedy Model Selection

In [28]:
def hill_climbing_ensemble(oof_dict, test_dict, y_true, max_models=50):
    """
    Greedy hill climbing: iteratively add the model (with replacement)
    that most improves the ensemble AUC.
    """
    model_names = list(oof_dict.keys())
    n = len(y_true)

    best_ensemble_oof = np.zeros(n)
    best_ensemble_test = np.zeros(len(test))
    selected = []
    best_score = 0

    for step in range(max_models):
        best_name = None
        best_step_score = best_score

        for name in model_names:
            trial_oof = (best_ensemble_oof * len(selected) + rank_avg(oof_dict[name])) / (len(selected) + 1)
            trial_score = roc_auc_score(y_true, trial_oof)
            if trial_score > best_step_score:
                best_step_score = trial_score
                best_name = name

        if best_name is None:
            print(f"  Hill climbing stopped at step {step} — no improvement")
            break

        selected.append(best_name)
        best_ensemble_oof = (best_ensemble_oof * (len(selected) - 1) + rank_avg(oof_dict[best_name])) / len(selected)
        best_ensemble_test = (best_ensemble_test * (len(selected) - 1) + rank_avg(test_dict[best_name])) / len(selected)
        best_score = best_step_score

        if step < 15 or step % 10 == 0:
            print(f"  Step {step:3d}: +{best_name:15s} -> AUC = {best_score:.6f}")

    # Count selections
    from collections import Counter
    counts = Counter(selected)
    print(f"\n  Final Hill Climbing AUC: {best_score:.6f}")
    print(f"  Models selected ({len(selected)} total):")
    for name, cnt in counts.most_common():
        print(f"    {name:15s}: {cnt:3d} times ({cnt/len(selected)*100:.1f}%)")

    return best_ensemble_oof, best_ensemble_test, best_score

hc_oof, hc_test, hc_score = hill_climbing_ensemble(oof_dict, test_dict, train_full[TARGET], max_models=50)

  Step   0: +XGB_v2          -> AUC = 0.915824
  Step   1: +LGB_v2          -> AUC = 0.915859
  Step   2: +CAT_v1          -> AUC = 0.915878
  Step   3: +XGB_v2          -> AUC = 0.915900
  Step   4: +XGB_v2          -> AUC = 0.915902
  Hill climbing stopped at step 5 — no improvement

  Final Hill Climbing AUC: 0.915902
  Models selected (5 total):
    XGB_v2         :   3 times (60.0%)
    LGB_v2         :   1 times (20.0%)
    CAT_v1         :   1 times (20.0%)


## 8. Final Ensemble Selection & Submission

In [29]:
# ============ COMPARE ALL ENSEMBLES ============
print("\n" + "="*60)
print("FINAL ENSEMBLE COMPARISON")
print("="*60)

ensembles = {
    "Equal Rank Avg": (ensemble_oof, ensemble_test),
    "Optimized Weighted": (opt_oof, opt_test),
    "LR Meta-Model": (lr_meta_oof, lr_meta_test),
    "Ridge Meta-Model": (ridge_meta_oof, ridge_meta_test),
    "LGB Meta-Model": (lgb_meta_oof, lgb_meta_test),
    "Hill Climbing": (hc_oof, hc_test),
}

best_name = None
best_score = 0
best_test = None

for name, (oof_p, test_p) in ensembles.items():
    score = roc_auc_score(train_full[TARGET], oof_p)
    print(f"  {name:25s} CV AUC = {score:.6f}")
    if score > best_score:
        best_score = score
        best_name = name
        best_test = test_p

print(f"\n🏆 BEST: {best_name} with CV AUC = {best_score:.6f}")
print("="*60)


FINAL ENSEMBLE COMPARISON
  Equal Rank Avg            CV AUC = 0.915674
  Optimized Weighted        CV AUC = 0.915889
  LR Meta-Model             CV AUC = 0.915902
  Ridge Meta-Model          CV AUC = 0.915902
  LGB Meta-Model            CV AUC = 0.915789
  Hill Climbing             CV AUC = 0.915902

🏆 BEST: Hill Climbing with CV AUC = 0.915902


In [30]:
# --- Also make a blend of the best ensembles ---
final_blend_oof = (
    0.3 * rank_avg(opt_oof) +
    0.3 * rank_avg(hc_oof) +
    0.2 * rank_avg(lr_meta_oof) +
    0.2 * rank_avg(lgb_meta_oof)
)
final_blend_test = (
    0.3 * rank_avg(opt_test) +
    0.3 * rank_avg(hc_test) +
    0.2 * rank_avg(lr_meta_test) +
    0.2 * rank_avg(lgb_meta_test)
)
blend_score = roc_auc_score(train_full[TARGET], final_blend_oof)
print(f"Final Super-Blend CV AUC: {blend_score:.6f}")

# Use the absolute best
if blend_score > best_score:
    best_test = final_blend_test
    best_score = blend_score
    best_name = "Super-Blend"
    print(f"🏆 Super-Blend wins with CV AUC = {best_score:.6f}")
else:
    print(f"Keeping {best_name} as final submission")

Final Super-Blend CV AUC: 0.915907
🏆 Super-Blend wins with CV AUC = 0.915907


In [31]:
# ============ GENERATE SUBMISSION ============
sub = pd.DataFrame({
    "id": test["id"],
    "Churn": best_test
})

sub.to_csv("submission.csv", index=False)
print(f"\n✅ Submission saved: submission.csv")
print(f"   Shape: {sub.shape}")
print(f"   Prediction stats:")
print(f"   Mean: {sub['Churn'].mean():.4f}")
print(f"   Std:  {sub['Churn'].std():.4f}")
print(f"   Min:  {sub['Churn'].min():.4f}")
print(f"   Max:  {sub['Churn'].max():.4f}")
print(sub.head())


✅ Submission saved: submission.csv
   Shape: (254655, 2)
   Prediction stats:
   Mean: 0.5000
   Std:  0.2886
   Min:  0.0007
   Max:  0.9998
       id     Churn
0  594194  0.498304
1  594195  0.011394
2  594196  0.553029
3  594197  0.176616
4  594198  0.804899


In [32]:
# ============ ALSO SAVE ALTERNATIVE SUBMISSIONS ============
for ens_name, (_, test_p) in ensembles.items():
    fname = f"submission_{ens_name.lower().replace(' ', '_').replace('-', '_')}.csv"
    pd.DataFrame({"id": test["id"], "Churn": test_p}).to_csv(fname, index=False)
    print(f"Saved: {fname}")

print("\n🎯 All submissions saved. Upload the best one to Kaggle!")

Saved: submission_equal_rank_avg.csv
Saved: submission_optimized_weighted.csv
Saved: submission_lr_meta_model.csv
Saved: submission_ridge_meta_model.csv
Saved: submission_lgb_meta_model.csv
Saved: submission_hill_climbing.csv

🎯 All submissions saved. Upload the best one to Kaggle!
